## IE Results Viewer
Viewer for Information Extraction results on OASIS journal reports. Facilitates interactive adjustment of section and significance scores to affect results ranking.

In [1]:
import math
import pandas as pd
import datetime, io, os, base64, json
from ATRIUM_T4_1_2_IE_results_utils import *
from ipywidgets import Button, Output, FloatSlider, Layout, HBox, VBox, FileUpload, widgets
from IPython.display import display, HTML

# create UI controls and output areas for the interactive widget

# component for config file selection
config_file_upload = FileUpload(
    button_style='primary',
    description="Select config file",  # Button text
    accept='.json',  # Accepted file extension
    multiple=False,  # True to accept multiple files upload else False
    layout=Layout(width='200px')
)
# output area to display the selected config file name
config_file_name_display = Output()


# component for input data file selection
input_file_upload = FileUpload(
    button_style='primary',
    description="Select input file",  # Button text
    accept='.json',  # Accepted file extension
    multiple=False,  # True to accept multiple files upload else False
    layout=Layout(width='200px')
)
# output area to display the selected input file name
input_file_name_display = Output()

# sliders for adjusting relevance scoring parameters
s_style = {'description_width': '150px', 'handle_color': 'lightblue'}
s_layout=Layout(width='500px')

# title score slider
slider_t = FloatSlider(
    description='Title score', 
    value=DEFAULT_CONFIG.get("span_scorer", {}).get("sec_scores", {}).get("title", 0.0), 
    min=0.0, 
    max=50.0, 
    step=0.1, 
    layout=s_layout, 
    style=s_style, 
    tooltip="Score for terms appearing in the title section"
)

# abstract score slider
slider_a = FloatSlider(
    description='Abstract score', 
    value=DEFAULT_CONFIG.get("span_scorer", {}).get("sec_scores", {}).get("abstract", 0.0), 
    min=0.0, 
    max=50.0, 
    step=0.1, 
    layout=s_layout, 
    style=s_style, 
    tooltip="Score for terms appearing in the abstract section"
)

# body score slider
slider_b = FloatSlider(
    description='Body score', 
    value=DEFAULT_CONFIG.get("span_scorer", {}).get("sec_scores", {}).get("body", 0.0), 
    min=0.0, 
    max=50.0, 
    step=0.1, 
    layout=s_layout, 
    style=s_style, 
    tooltip="Score for terms appearing in the body section"
)

# significance score slider
slider_s = FloatSlider(
    description='Significance score', 
    value=DEFAULT_CONFIG.get("span_scorer", {}).get("sig_score", 0.0), 
    min=0.0, 
    max=50.0, 
    step=0.1, 
    layout=s_layout, 
    style=s_style, 
    tooltip="Score for terms close to significance terms"
)

# minimum relevance score (MRS) slider
slider_mrs = FloatSlider(
    description='Minimum score', 
    value=DEFAULT_CONFIG.get("span_scorer", {}).get("min_relevance_score", 0.0), 
    min=0.0, 
    max=50.0, 
    step=0.1, 
    layout=s_layout, 
    style=s_style, 
    tooltip="Minimum relevance score for a term to be included in the results")

# minimum relevance count (MRC) slider
slider_mrc = FloatSlider(
    description='Minimum count', 
    value=DEFAULT_CONFIG.get("span_scorer", {}).get("min_relevance_count", 1), 
    min=1, 
    max=50, 
    step=1, 
    layout=s_layout, 
    style=s_style, 
    tooltip="Minimum relevance count for a term to be included in the results")


# Refresh / Save / Reset buttons
b_layout = Layout(width='200px')
btn_refresh = Button(description="Refresh results", icon='refresh', button_style='primary', layout=b_layout)
btn_save = Button(description="Save config", icon='save', button_style='primary', layout=b_layout)
btn_reset = Button(description="Clear and reset", icon='times', button_style='primary', layout=b_layout)
# UI output for results table
outputs = Output()

# arrange and display all UI components
sliders = VBox([slider_t, slider_a, slider_b, slider_s, slider_mrs, slider_mrc], layout=Layout(display='flex', flex_flow='column', gap='5px'))
buttons = HBox([btn_refresh, btn_save, btn_reset], layout=Layout(display='flex', flex_flow='row', gap='10px'))

display(HBox([config_file_upload, config_file_name_display]))
display(HBox([input_file_upload, input_file_name_display]))
display(VBox([sliders, buttons, outputs]))


# when a config file is selected display the selected file name 
def on_config_file_upload_change(change):
    config_file_name_display.clear_output()
    outputs.clear_output()    
    if config_file_upload.value:
        config_file = config_file_upload.value[0]
        s = f"Selected: {config_file.get('name', '-')}"
        with config_file_name_display:
            display(HTML(s))    
        config_data = json.load(io.BytesIO(config_file['content'])) 
        set_slider_values(config_data)  # update sliders with config values
config_file_upload.observe(on_config_file_upload_change, names='value')


# when an input file is selected display the selected file name 
def on_input_file_upload_change(change):
    input_file_name_display.clear_output()
    outputs.clear_output()    
    if input_file_upload.value:
        input_file = input_file_upload.value[0]
        s = f"Selected: {input_file.get('name', '-')}"
        with input_file_name_display:
            display(HTML(s))    
input_file_upload.observe(on_input_file_upload_change, names='value')


def set_slider_values(config: dict):
    sec_scores = (config or {}).get("span_scorer", {}).get("sec_scores", {})
    slider_t.value = sec_scores.get("title", DEFAULT_CONFIG.get("span_scorer", {}).get("sec_scores", {}).get("title"))
    slider_a.value = sec_scores.get("abstract", DEFAULT_CONFIG.get("span_scorer", {}).get("sec_scores", {}).get("abstract"))
    slider_b.value = sec_scores.get("body", DEFAULT_CONFIG.get("span_scorer", {}).get("sec_scores", {}).get("body"))
    slider_s.value = (config or {}).get("span_scorer", {}).get("sig_score", DEFAULT_CONFIG.get("sig_score"))
    slider_mrs.value = (config or {}).get("min_relevance_score", DEFAULT_CONFIG.get("min_relevance_score"))
    slider_mrc.value = (config or {}).get("min_relevance_count", DEFAULT_CONFIG.get("min_relevance_count"))
    

def save_config_values(file_name: str):

    # modify existing config values or create new config data
    config_data = {}
    if config_file_upload.value:
        config_file = config_file_upload.value[0]
        config_data = json.load(io.BytesIO(config_file['content'])) 

    # Truncate value to specified decimal places without rounding
    truncate = lambda x, n: math.floor(x * 10 ** n) / 10 ** n  

    # new config data structure to be saved    
    new_config = {
        "span_scorer": {
            "sec_scores": {
                "title": truncate(slider_t.value, 2),
                "abstract": truncate(slider_a.value, 2),
                "body": truncate(slider_b.value, 2)
            },
            "sig_score": truncate(slider_s.value, 2)
        },
        "min_relevance_score": truncate(slider_mrs.value, 2),
        "min_relevance_count": math.floor(slider_mrc.value)
    }
    # override existing values with current slider values
    config_data.update(new_config)

    # Encode the fresh string data into base64 bytes
    b64_data = base64.b64encode(json.dumps(config_data).encode()).decode()
    
    # 3. Inject a hidden link to trigger its own click event in the browser
    js_download_trigger = f"""
        <a id="temp_dl_link" download="{file_name}" href="data:text/plain;base64,{b64_data}" style="display:none;"></a>
        <script>
            var link = document.getElementById('temp_dl_link');
            link.click();
            link.remove(); // Clean up the page source immediately
        </script>
        """
    display(HTML(js_download_trigger))


# refresh results when refresh button is clicked
def on_refresh_click(b):
    outputs.clear_output()
    input_file = input_file_upload.value[0] if input_file_upload.value else None
    if input_file:
        df = pd.DataFrame()        
        input_file_type = input_file.get('type','').strip().lower() 
        input_file_ext =  os.path.splitext(input_file.get('name','').strip().lower())             
        if input_file_type == "application/json" or input_file_ext == "json":  
            #data = json.load(io.StringIO(input_file['content'])) 
            data = json.load(io.BytesIO(input_file['content'])) 
            df = pd.DataFrame(data.get('spans', []))            
        
        # set any NaN values to blank string      
        #df.fillna("", inplace=True)  
        input_data = df.to_dict(orient='records')   
                        
        df = aggregate_results_by_concept(input_data, slider_t, slider_a, slider_b, slider_s) # aggregated results by concept id
        df = df[df['score'] >= slider_mrs.value]  # filtering by minimum relevance score
        df = df[df['count'] >= slider_mrc.value]  # filtering by minimum relevance count
        df = df.sort_values(by='score', ascending=False) #.head(20)
        df = df[['span', 'label', 'count', 'sec_score', 'sig_score', 'score']]
        styled_df = (df.style
            .set_caption("Results aggregated by concept")
            #.hide(subset=['id', 'text'], axis=1)
            .hide(axis="index")
            .highlight_max(subset=['sig_score', 'count']) #, color='red'
            .background_gradient(subset=['score', 'sec_score', 'sig_score', 'count']) #, cmap='YlOrRd', low=0.2, high=0.8
             #.applymap(color_negative_red)
            .format(na_rep="n/a")
            .format( "{:.2f}", subset=['sec_score', 'sig_score', 'score']))
       
        with outputs:
            display(HTML(styled_df.to_html(index=False)))   
btn_refresh.on_click(on_refresh_click)


def on_save_click(b):
    config_file_name = config_file_upload.value[0]['name'] if config_file_upload.value else f"config_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    save_config_values(config_file_name)    
btn_save.on_click(on_save_click)


# clear fields and reset sliders when reset button is clicked
def on_reset_click(b):
    set_slider_values(DEFAULT_CONFIG)
    outputs.clear_output()
    config_file_upload.value = []
    input_file_upload.value = []
    config_file_name_display.clear_output()
    input_file_name_display.clear_output()
btn_reset.on_click(on_reset_click)